# 小鴨動作量測 v3（A100）/ Duck Core physics sweep v3

用**官方** microduck_rl 的物理環境（mjlab + MuJoCo Warp）和**官方 v5 神經網路**，同時模擬數千隻小鴨。指令的處理完全照真機的 robotd（頭部平滑、動作比例與濾波），隨機變化和 v2 一樣（官方訓練最後階段的範圍）。

**這一版（v3）量四件事**，一格跑完（第 6 節），寫成一個檔 `expression-sweep-v3.json`，旁邊附每一隻鴨的原始紀錄 `expression-sweep-v3.trials.jsonl`（任何人都能重算）：

1. **坐 ↔ 站分段計數**：站起、站著、坐下 3 秒內、坐著，各自數跌倒。主要用 robotd 自己的跌倒判斷（傾斜超過 60° 持續 0.2 秒），舊的「超過 45°」另列一欄；坐穩前就歪的另計，不算跌倒。每種物理 4096 隻。
2. **跌倒後自己爬起來**：跌倒後讓走路網路在零速度下自己起來，看 8 秒內、停 2 秒、再 8 秒內有沒有站穩（傾斜 < 25° 維持 1 秒）。
3. **看的姿勢**：`look.glance_down`（半低頭）、`look.up`（抬頭）、`look.seated`（坐著看），站著和坐著各跑，量跌倒、傾斜、漂移，還有鏡頭實際指到幾度。
4. **低頭走路**：vx 0.30 直走 6 秒，頭擺正 vs 半低頭，比速度和偏航。

**開始前**：`執行階段 → 變更執行階段類型 → A100 GPU`。然後 `執行階段 → 全部執行`。

**時間**：安裝約 10 分鐘，小跑確認約 3–7 分鐘，v3 量測約 7 分鐘（由 v2 的 519 秒換算：5.5–7.3 分鐘）。整段約 20–25 分鐘，約 2.3–2.9 個運算單元。跑完記得 `執行階段 → 中斷連線並刪除執行階段`，否則會一直扣運算單元。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 參數 / Parameters

In [ ]:
DR = "final"               # 官方訓練最後階段的隨機化範圍（和 v2 一樣）
ENVS = 8192                # 一批同時模擬幾隻小鴨
LOOK_RANDOMIZATIONS = 96   # 每個「看」的組合跑幾隻隨機鴨
POSTURE_EPISODES = 4096    # 坐↔站：每種物理幾隻鴨（至少 256）
HEADWALK_EPISODES = 512    # 低頭走路：每種頭的姿勢幾隻鴨
SAVE_TO_DRIVE = False      # 主人規定不掛載 Drive：結果只用瀏覽器下載（改 True 才會存到 Google Drive，需要授權）
RUN_CLIP_SWEEP = False     # （可選）第 7 節：重跑全部表情的力道掃描（v2 那種，約多 8–12 分鐘）
RANDOMIZATIONS = 32        # 第 7 節用：每個表情組合跑幾次

## 2. 程式碼 / Code

In [ ]:
# ── 取得程式碼 / Get the code ─────────────────────────────────────────
# 方法 A：Colab「密鑰」(左側鑰匙圖示) 新增 GITHUB_TOKEN（只給 duckmind 讀取權限）
# 方法 B：不設密鑰時，執行後會請你上傳 duckmind-colab.zip
import os, subprocess, zipfile, pathlib
BRANCH = "feature/a100-expression-name"
REPO = "github.com/<owner>/duckmind.git"  # 你的 duckmind repo / your duckmind repo
root = pathlib.Path("/content")
os.chdir(root)
token = None
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    pass
if not (root / "duckmind").exists():
    if token:
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                        f"https://{token}@{REPO}", "duckmind"], check=True)
    else:
        from google.colab import files
        print("請選擇 duckmind-colab.zip 上傳 / upload duckmind-colab.zip")
        uploaded = files.upload()
        name = next(iter(uploaded))
        with zipfile.ZipFile(name) as archive:
            archive.extractall(root)
print(sorted(p.name for p in (root / "duckmind").iterdir())[:20])

In [ ]:
# 官方 microduck_rl，鎖定在 duckmind 驗證過的版本
MICRODUCK_RL_COMMIT = "cb70b792312d559a4da09064d92009079671815f"
if not (root / "microduck_rl").exists():
    subprocess.run(["git", "clone", "https://github.com/pollen-robotics/microduck_rl.git"], check=True)
subprocess.run(["git", "-C", "microduck_rl", "checkout", "-q", MICRODUCK_RL_COMMIT], check=True)
!git -C microduck_rl log -1 --oneline

## 3. 安裝官方環境 / Install (uv, Python 3.12, locked versions)

In [ ]:
!pip -q install uv
!cd microduck_rl && uv sync --locked 2>&1 | tail -3
!cd microduck_rl && uv pip install onnxruntime 2>&1 | tail -1
PY = "/content/microduck_rl/.venv/bin/python"
!{PY} -c "import torch, mjlab; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"

import collections, os, re, subprocess


def sweep(*args):
    """Run duckmind's sweep runner; show its own log lines (and any failure), stop on error."""
    env = dict(os.environ, PYTHONPATH="/content/duckmind/src", WANDB_MODE="disabled")
    cmd = [PY, "-m", "duckmind.expression_sweep.runner", "--policies", "policies",
           "--device", "cuda:0", *map(str, args)]
    tail, showing = collections.deque(maxlen=40), False
    with subprocess.Popen(cmd, cwd="/content", env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            tail.append(line)
            ours = re.match(r"^\[\s*\d+\.\ds\]", line) is not None
            if ours:
                showing = "failed:" in line
            if ours or showing:
                print(line, end="", flush=True)
    if proc.returncode:
        print("".join(tail))
        raise RuntimeError(f"sweep runner exited with {proc.returncode}")

## 4. 官方 v5 神經網路 / Official v5 policies (Hugging Face)

In [ ]:
import hashlib, urllib.request
EXPECTED = {
    "velstand.onnx": "1c659be55da94bc5753b707de5c6a3e7c49931e05ca3b6991615cef1a8ba9a45",
    "alpha_sitstand.onnx": "c6c40e35e726eabd803d633e090d112994f469921152448367953fbaf9799bc8",
    "manifest.json": "622048c2c23ea58942023f66fd16b189a875fd169e88d85beb16ebbe63b20c94",
}
policies = root / "policies"
policies.mkdir(exist_ok=True)
for name, digest in EXPECTED.items():
    target = policies / name
    if not target.exists():
        urllib.request.urlretrieve(f"https://huggingface.co/pollen-robotics/microduck-policies/resolve/v5/{name}", target)
    got = hashlib.sha256(target.read_bytes()).hexdigest()
    print(name, "OK" if got == digest else f"MISMATCH {got}")
    assert got == digest, "v5 檔案與 duckmind 驗證的版本不同，先停下來"

## 5. 小跑確認 / Smoke test (~3–7 min)

In [ ]:
# 每一種新量測都用很少的鴨跑一次，確定程式在 GPU 上沒問題，才跑完整的。
import json, pathlib
sweep("--out", "smoke-v3.json", "--envs", 256, "--scenarios", "look,posture,headwalk",
      "--look-randomizations", 1, "--posture-episodes", 64, "--headwalk-episodes", 32)
smoke = json.loads(pathlib.Path("/content/smoke-v3.json").read_text())
print("小跑完成:", smoke["complete"], " 錯誤:", [e["phase"] for e in smoke["errors"]])
assert smoke["complete"], "小跑有錯，先停下來（錯誤訊息在上面）"

## 6. v3 量測（這一格跑完全部）/ v3 measurements — one cell

跑完會印出重點，並把 `expression-sweep-v3.json` 和 `expression-sweep-v3.trials.jsonl` 存到 Drive 並下載。

In [ ]:
import hashlib, json, pathlib, sys
from google.colab import files
OUT = pathlib.Path("/content/expression-sweep-v3.json")
TRIALS = OUT.with_suffix(".trials.jsonl")
if SAVE_TO_DRIVE:  # 先授權，跑完才不會停下來等
    from google.colab import drive
    drive.mount("/content/drive")

sweep("--out", OUT, "--envs", ENVS, "--dr", DR, "--scenarios", "look,posture,headwalk",
      "--look-randomizations", LOOK_RANDOMIZATIONS, "--posture-episodes", POSTURE_EPISODES,
      "--headwalk-episodes", HEADWALK_EPISODES, "--microduck-rl", "microduck_rl")

sys.path.insert(0, "/content/duckmind/src")
from duckmind.expression_sweep import envelope, summary
result = envelope.load(OUT)
print(f"\n完成: {result['complete']}  錯誤: {[e['phase'] for e in result['errors']]}  "
      f"模擬 {result['source']['seconds']} 秒（預估 {result['plan']['estimate_s']}）")


def pct(p):
    if not p["n"]:
        return "n=0"
    lo, hi = p["wilson95"]
    return f"{p['count']}/{p['n']} = {100 * p['rate']:.2f}% [95% {100 * lo:.2f}–{100 * hi:.2f}%]"


for task, stats in result["motion"].get("posture", {}).items():
    name = "坐站網路的物理" if "SitStand" in task else "走路網路的物理"
    print(f"\n== 坐 → 站起 → 站 8 秒 → 坐下（{name}；坐穩的鴨 {stats['settled']} 隻）==")
    print("  坐穩前就歪（另計，不算跌倒）:", pct(stats["settle_tilt"]))
    for phase, zh in (("rise", "站起"), ("stand", "站著"), ("sit", "坐下 3 秒內"), ("seated", "之後坐著")):
        p = stats["phases"][phase]
        print(f"  {zh}跌倒  robotd 規則 {pct(p['robotd'])}   舊 45° {pct(p['tilt45'])}")
    r = stats["righting"]
    print(f"  跌倒 {r['falls']} 次 → 8 秒內自己起來 {pct(r['try1'])}；18 秒內 {pct(r['by_try2'])}")

print("\n== 看的姿勢（設計幅度；鏡頭角度: + 往上）==")
for clip, by_posture in result.get("looks_summary", {}).items():
    for posture, b in by_posture.items():
        shots = " / ".join(f"{c['pitch_deg']['mean']:+.1f}°（穩 {c['still']['count']}/{c['still']['n']}）"
                           for c in b["captures"])
        gate = b.get("gate", {}).get("passes")
        print(f"  {clip:18s} {'坐' if posture == 'sit' else '站'}  跌倒 {pct(b['falls_robotd'])}  "
              f"漂移 p95 {b['drift_m'].get('p95')} m  鏡頭 {shots}  過關 {gate}")

hw = result.get("head_walk", {})
for label, zh in (("level", "頭擺正"), ("half_down", "半低頭")):
    if label in hw:
        b = hw[label]
        print(f"\n  {zh}走路: 速度 {b['speed_mps'].get('mean')} m/s（後半 {b['steady_speed_mps'].get('mean')}）"
              f" 偏航 {b['yaw_rate_dps'].get('mean')} °/s  跌倒 {pct(b['falls_robotd'])}")
if "compare_to_level" in hw:
    print("  半低頭 − 擺正:", hw["compare_to_level"]["half_down"]["speed_mps"])

rebuilt = summary.recompute(TRIALS)
same = rebuilt.get("looks_summary") == result.get("looks_summary") and rebuilt.get("head_walk") == result.get("head_walk")
same = same and all(result["motion"]["posture"][task][key] == value
                    for task, block in rebuilt.get("posture", {}).items() for key, value in block.items())
print("\n原始逐筆紀錄可以重算出一樣的摘要:", same)
print("trials sha256 一致:", hashlib.sha256(TRIALS.read_bytes()).hexdigest() == result["source"]["trials"]["sha256"])

if SAVE_TO_DRIVE:
    dest = pathlib.Path("/content/drive/MyDrive/duckmind-sweeps")
    dest.mkdir(parents=True, exist_ok=True)
    for p in (OUT, TRIALS):
        (dest / p.name).write_bytes(p.read_bytes())
    print("已存到", dest)
files.download(str(OUT))
files.download(str(TRIALS))

## 7.（可選）重跑表情力道掃描 / Optional: the expression clip sweep (v2 style)

只有在 `RUN_CLIP_SWEEP = True` 時才跑。會掃描**現在** Duck Core 全部的表情（v2 之後新增的 12 個也一起），寫出 `sweep.json` 和完整的 `expression_caps.json`。Duck Core 的表情還在改的時候先不要跑。

In [ ]:
if RUN_CLIP_SWEEP:
    sweep("--out", "sweep.json", "--envs", ENVS, "--randomizations", RANDOMIZATIONS, "--dr", DR,
          "--scenarios", "head", "--microduck-rl", "microduck_rl")
else:
    print("略過（RUN_CLIP_SWEEP = False）")

In [ ]:
if RUN_CLIP_SWEEP:
    import json, pathlib, sys
    sys.path.insert(0, "/content/duckmind/src")
    from duckmind.expression_sweep import envelope
    result = envelope.load(pathlib.Path("/content/sweep.json"))
    print("跑在目前的表情表上:", envelope.is_current(result))
    capped = envelope.caps(result)
    for clip, by_posture in sorted(capped.items()):
        print(f"{clip:16s} 坐 {by_posture['sit']:.2f}  站 {by_posture['stand']:.2f}  做到幾成(坐/站) "
              f"{envelope.fidelity(result, clip, 'sit') or 0:.2f}/{envelope.fidelity(result, clip, 'stand') or 0:.2f}")
    envelope.write_caps(result, pathlib.Path("/content/expression_caps.json"))
    from google.colab import files
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        dest = pathlib.Path("/content/drive/MyDrive/duckmind-sweeps")
        dest.mkdir(parents=True, exist_ok=True)
        for name in ("sweep.json", "sweep.trials.jsonl", "expression_caps.json"):
            (dest / name).write_bytes((pathlib.Path("/content") / name).read_bytes())
    for name in ("expression_caps.json", "sweep.json", "sweep.trials.jsonl"):
        files.download(f"/content/{name}")

跑完請 **中斷連線並刪除執行階段**。把 `expression-sweep-v3.json` 和 `expression-sweep-v3.trials.jsonl`（有跑第 7 節的話再加 `sweep.json`、`sweep.trials.jsonl`、`expression_caps.json`）交給 duckmind，放到 `docs/validation/evidence/`。